# Setup: Deploy Sample Apps with Source Code

This notebook uploads sample app source code to workspace and deploys it to existing sample apps.

## Configuration

In [ ]:
import os

try:
    catalog = dbutils.widgets.get("catalog")
except:
    catalog = "YOUR_SOURCE_CATALOG"

try:
    schema = dbutils.widgets.get("schema")
except:
    schema = "apps_demo"

volume_path = f"/Volumes/{catalog}/{schema}/apps_migration"
user_email = spark.sql("SELECT current_user()").collect()[0][0]
apps_source_path = f"/Workspace/Users/{user_email}/sample_apps_source"

print(f"Volume path: {volume_path}")
print(f"User: {user_email}")
print(f"Apps source path: {apps_source_path}")

## Initialize SDK

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ImportFormat
import requests
import base64
import time
from io import BytesIO

w = WorkspaceClient()
workspace_url = w.config.host.rstrip('/')
headers = w.config.authenticate()
print(f"Connected to: {workspace_url}")

## Define sample app sources

In [ ]:
gradio_app_py = """import gradio as gr

def greet(name):
    return f"Hello, {name}! Welcome to the Gradio app."

demo = gr.Interface(
    fn=greet,
    inputs=gr.Textbox(label="Your Name"),
    outputs=gr.Textbox(label="Greeting"),
    title="Sample Gradio App",
    description="A simple greeting app for migration testing."
)

if __name__ == "__main__":
    demo.launch(server_name="0.0.0.0", server_port=8000)
"""

gradio_app_yaml = """command:
  - python
  - app.py
env:
  - name: APP_NAME
    value: sample-gradio-app
"""

streamlit_app_py = """import streamlit as st

st.set_page_config(page_title="Sample Streamlit App", page_icon=":rocket:")

st.title("Sample Streamlit App")
st.write("This is a sample Streamlit app for migration testing.")

name = st.text_input("Enter your name:")
if name:
    st.success(f"Hello, {name}! Welcome to Streamlit.")

st.metric("Migration Status", "Ready", "+1")
"""

streamlit_app_yaml = """command:
  - streamlit
  - run
  - app.py
  - --server.port
  - "8000"
  - --server.address
  - "0.0.0.0"
env:
  - name: APP_NAME
    value: sample-streamlit-app
"""

dash_app_py = """from dash import Dash, html, dcc

app = Dash(__name__)

app.layout = html.Div([
    html.H1("Sample Dash App"),
    html.P("A simple Dash application for migration testing."),
    dcc.Input(id="name-input", type="text", placeholder="Enter your name"),
    html.Div(id="greeting-output")
])

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=8000, debug=False)
"""

dash_app_yaml = """command:
  - python
  - app.py
env:
  - name: APP_NAME
    value: sample-dash-app
"""

sample_apps = {
    "sample-gradio-app": {
        "app.py": gradio_app_py,
        "requirements.txt": "gradio>=4.0.0\n",
        "app.yaml": gradio_app_yaml
    },
    "sample-streamlit-app": {
        "app.py": streamlit_app_py,
        "requirements.txt": "streamlit>=1.30.0\n",
        "app.yaml": streamlit_app_yaml
    },
    "sample-dash-app": {
        "app.py": dash_app_py,
        "requirements.txt": "dash>=2.14.0\n",
        "app.yaml": dash_app_yaml
    }
}

print(f"Defined {len(sample_apps)} sample apps")

## Upload source code to workspace

In [ ]:
def create_workspace_folder(path):
    resp = requests.post(
        f"{workspace_url}/api/2.0/workspace/mkdirs",
        headers=headers,
        json={"path": path}
    )
    return resp.status_code in [200, 201]

def upload_workspace_file(path, content):
    resp = requests.post(
        f"{workspace_url}/api/2.0/workspace/import",
        headers=headers,
        json={
            "path": path,
            "content": base64.b64encode(content.encode('utf-8')).decode('utf-8'),
            "format": "AUTO",
            "overwrite": True
        }
    )
    return resp.status_code in [200, 201]

print("Creating workspace folders and uploading source files...")

for app_name, files in sample_apps.items():
    app_path = f"{apps_source_path}/{app_name}"
    print(f"\nApp: {app_name}")
    
    create_workspace_folder(app_path)
    print(f"  Created folder: {app_path}")
    
    for filename, content in files.items():
        file_path = f"{app_path}/{filename}"
        if upload_workspace_file(file_path, content):
            print(f"  Uploaded: {filename}")
        else:
            print(f"  FAILED: {filename}")

print("\nSource files uploaded!")

## Deploy source code to apps

In [ ]:
def deploy_app(app_name, source_path):
    """Deploy source code to an existing app."""
    resp = requests.post(
        f"{workspace_url}/api/2.0/apps/{app_name}/deployments",
        headers=headers,
        json={
            "source_code_path": source_path,
            "mode": "AUTO_SYNC"
        }
    )
    if resp.status_code in [200, 201]:
        return resp.json()
    else:
        raise Exception(f"Deploy failed: {resp.text}")

print("Deploying apps...")
print("=" * 60)

deployments = []
for app_name in sample_apps.keys():
    source_path = f"{apps_source_path}/{app_name}"
    print(f"\nDeploying {app_name}...")
    try:
        result = deploy_app(app_name, source_path)
        deployment_id = result.get('deployment_id', 'unknown')
        print(f"  Deployment started: {deployment_id}")
        deployments.append({"app": app_name, "deployment_id": deployment_id, "status": "STARTED"})
    except Exception as e:
        print(f"  FAILED: {e}")
        deployments.append({"app": app_name, "deployment_id": "", "status": "FAILED"})

print("\n" + "=" * 60)

## Wait for deployments

In [ ]:
print("Waiting for deployments to complete...")
time.sleep(30)

print("\nFinal app status:")
for app_name in sample_apps.keys():
    resp = requests.get(f"{workspace_url}/api/2.0/apps/{app_name}", headers=headers)
    if resp.status_code == 200:
        app = resp.json()
        compute = app.get('compute_status', {}).get('state', 'UNKNOWN')
        app_state = app.get('app_status', {}).get('state', 'UNKNOWN')
        deployment = app.get('active_deployment', {})
        source_path = deployment.get('source_code_path', 'No deployment')
        print(f"  {app_name}: compute={compute}, app={app_state}, source={source_path}")
    else:
        print(f"  {app_name}: Failed to get status")

print("\nSetup complete! Apps now have source code deployed.")